In [ ]:
from google.colab import files
uploaded = files.upload()

Saving clothing_size_extended_50000.csv to clothing_size_extended_50000.csv


In [ ]:

import pandas as pd
import numpy as np
import cv2
import mediapipe as mp
import math
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

#load dataset

df = pd.read_csv("clothing_size_extended_50000.csv")
print("✅ Dataset Loaded Successfully:", df.shape)

le_gender = LabelEncoder()
le_size = LabelEncoder()

df['Gender'] = le_gender.fit_transform(df['Gender'])
df['Size'] = le_size.fit_transform(df['Size'])

X = df.drop(columns=['Size'])
y = df['Size']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#train models
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

xgb = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42)
xgb.fit(X_train, y_train)

print("✅ Models Trained Successfully")

# ----------------------------
# STEP 3: Camera & Pose Detection
# ----------------------------
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=True)

# Capture image from webcam
cap = cv2.VideoCapture(0)
print("\n🎥 Please stand in front of the camera...")
ret, frame = cap.read()
cap.release()

if not ret:
    print("❌ Could not capture image from camera.")
    exit()

# Convert to RGB
image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
results = pose.process(image_rgb)

if results.pose_landmarks:
    landmarks = results.pose_landmarks.landmark

    # Helper function to calculate pixel distance
    def distance(p1, p2):
        return math.sqrt((p1.x - p2.x)**2 + (p1.y - p2.y)**2)

    # Shoulder width
    left_shoulder = landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER]
    right_shoulder = landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER]
    shoulder_width = distance(left_shoulder, right_shoulder) * 180  # approximate scaling to cm

    # Arm length (shoulder → wrist)
    left_wrist = landmarks[mp_pose.PoseLandmark.LEFT_WRIST]
    right_wrist = landmarks[mp_pose.PoseLandmark.RIGHT_WRIST]
    left_arm_length = distance(left_shoulder, left_wrist) * 180
    right_arm_length = distance(right_shoulder, right_wrist) * 180
    arm_length = (left_arm_length + right_arm_length)/2

    # Leg length (hip → ankle)
    left_hip = landmarks[mp_pose.PoseLandmark.LEFT_HIP]
    right_hip = landmarks[mp_pose.PoseLandmark.RIGHT_HIP]
    left_ankle = landmarks[mp_pose.PoseLandmark.LEFT_ANKLE]
    right_ankle = landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE]
    left_leg_length = distance(left_hip, left_ankle) * 180
    right_leg_length = distance(right_hip, right_ankle) * 180
    leg_length = (left_leg_length + right_leg_length)/2

    # Height estimation (approx.)
    top_head = landmarks[mp_pose.PoseLandmark.NOSE]  # rough approximation
    bottom_feet = (left_ankle.y + right_ankle.y)/2
    height = (bottom_feet - top_head.y) * 180

    print(f"\n📏 Estimated Measurements (approx.):")
    print(f"Height: {height:.2f} cm, Shoulder: {shoulder_width:.2f} cm, Arm: {arm_length:.2f} cm, Leg: {leg_length:.2f} cm")

    # ----------------------------
    # STEP 4: Optional User Inputs for Remaining Features
    # ----------------------------
    weight = float(input("Enter your Weight (kg): "))
    age = int(input("Enter your Age: "))
    gender_input = input("Enter Gender (Male/Female): ").strip()
    chest = float(input("Enter Chest Circumference (cm): "))
    waist = float(input("Enter Waist Circumference (cm): "))
    hip = float(input("Enter Hip Circumference (cm): "))

    gender_encoded = le_gender.transform([gender_input])[0]

    # ----------------------------
    # STEP 5: Predict Clothing Size
    # ----------------------------
    user_sample = pd.DataFrame({
        'Height_cm':[height],
        'Weight_kg':[weight],
        'Age':[age],
        'Gender':[gender_encoded],
        'Shoulder_cm':[shoulder_width],
        'Chest_cm':[chest],
        'Waist_cm':[waist],
        'Hip_cm':[hip],
        'Arm_length_cm':[arm_length],
        'Leg_length_cm':[leg_length]
    })

    rf_pred = le_size.inverse_transform(rf.predict(user_sample))[0]
    dt_pred = le_size.inverse_transform(dt.predict(user_sample))[0]
    xgb_pred = le_size.inverse_transform(xgb.predict(user_sample))[0]

    print("\n👕 Predicted Clothing Sizes:")
    print(f"Random Forest : {rf_pred}")
    print(f"Decision Tree : {dt_pred}")
    print(f"XGBoost       : {xgb_pred}")

else:
    print("❌ No human pose detected. Please try again and ensure full body is visible.")


✅ Dataset Loaded Successfully: (50000, 11)
   Height_cm  Weight_kg  Age  Gender  Shoulder_cm  Chest_cm  Waist_cm  Hip_cm  \
0      179.2       79.1   34    Male         44.6      89.6      74.6   103.3   
1      148.6       57.8   28  Female         34.7      84.9      62.6    79.8   
2      168.4       68.8   21    Male         40.3      95.3      72.0    84.1   
3      175.0       67.8   50    Male         46.5      94.8      76.0    90.0   
4      167.9       59.1   57    Male         39.2      81.1      74.3    84.5   

   Arm_length_cm  Leg_length_cm Size  
0           64.4           95.3    S  
1           53.4           79.4   XS  
2           58.7           92.6    M  
3           62.8           90.4    S  
4           57.9           88.1   XS   



/usr/local/lib/python3.12/dist-packages/xgboost/training.py:183: UserWarning: [17:11:38] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



📊 Random Forest Metrics:
  Accuracy : 100.00%
  Precision: 100.00%
  Recall   : 100.00%
  F1 Score : 100.00%
  Classification Report:
              precision    recall  f1-score   support

           L       1.00      1.00      1.00       577
           M       1.00      1.00      1.00      3090
           S       1.00      1.00      1.00      3676
          XL       1.00      1.00      1.00        24
          XS       1.00      1.00      1.00      2633

    accuracy                           1.00     10000
   macro avg       1.00      1.00      1.00     10000
weighted avg       1.00      1.00      1.00     10000


📊 Decision Tree Metrics:
  Accuracy : 100.00%
  Precision: 100.00%
  Recall   : 100.00%
  F1 Score : 100.00%
  Classification Report:
              precision    recall  f1-score   support

           L       1.00      1.00      1.00       577
           M       1.00      1.00      1.00      3090
           S       1.00      1.00      1.00      3676
          XL       1.00 